# CommuniQ — QLoRA Fine-tune on Colab (Unsloth + Llama 3.1 8B)

**Runtime:** GPU (T4 on free Colab, A100 on Colab Pro)  
**Estimated time:** ~20 min on T4 for 200 examples / 1 epoch  

### What this notebook does
1. Installs Unsloth + TRL
2. Mounts your Google Drive (to read training data and save the adapter)
3. Loads `meta-llama/Meta-Llama-3.1-8B-Instruct` in 4-bit
4. Applies QLoRA adapters
5. Trains for 1-3 epochs on your ChatML JSONL
6. Saves the LoRA adapter to Drive
7. (Optional) Merges and exports to GGUF for Ollama

---
### Before running
1. Upload your `train.jsonl` and `val.jsonl` to Google Drive at `MyDrive/communiq/data/`
2. Accept the Llama 3.1 license at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
3. Create a HuggingFace access token at https://huggingface.co/settings/tokens
4. Set `Runtime > Change runtime type > T4 GPU`

In [ ]:
# @title Step 0 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — change runtime to GPU')

In [ ]:
# @title Step 1 -- Install Unsloth + deps (takes ~3 min first time)
import subprocess, importlib.metadata, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
    else:
        print('OK:', cmd[:80])
    return result.returncode

# Use stable PyPI release -- bundles correct pinned transformers/trl/peft versions
run('pip install unsloth --quiet')
run('pip install --force-reinstall --no-cache-dir unsloth --quiet')

# Deliberately do NOT `import unsloth` in this same cell/kernel session.
# The reinstall above can upgrade numpy (a transitive dep) to a version
# different from whatever numpy build Colab's kernel already has loaded in
# memory. numpy's C extensions can't be hot-swapped, so importing unsloth
# right after installing it can crash with:
#   RuntimeError: numpy was upgraded mid-session ... restart your runtime
# Checking the installed version via importlib.metadata (no import) avoids
# triggering that entirely.
print('All deps installed. Installed unsloth version:', importlib.metadata.version('unsloth'))
print('RESTARTING THE COLAB RUNTIME NOW -- this is expected.')
print('After it restarts, continue running from Step 2 onward.')
print('Do NOT re-run this Step 1 cell.')
os.kill(os.getpid(), 9)

In [ ]:
# @title Step 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/communiq/data'
OUTPUT_DIR = '/content/drive/MyDrive/communiq/lora_output'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = f'{DATA_DIR}/train.jsonl'
VAL_PATH   = f'{DATA_DIR}/val.jsonl'

print(f'Train: {TRAIN_PATH} — exists: {os.path.exists(TRAIN_PATH)}')
print(f'Val:   {VAL_PATH} — exists: {os.path.exists(VAL_PATH)}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# @title Step 3 — HuggingFace login (needed for Llama 3.1)
# Get your token from https://huggingface.co/settings/tokens
# Make sure you've accepted the Llama 3.1 license first.
from huggingface_hub import login
from google.colab import userdata

try:
    # If you saved HF_TOKEN in Colab secrets (recommended)
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in via Colab secret HF_TOKEN')
except Exception:
    # Fallback: manual paste
    hf_token = input('Paste your HuggingFace token: ').strip()
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in.')

In [ ]:
# @title Step 4 — Load base model in 4-bit with Unsloth
from unsloth import FastLanguageModel
import torch

# ── Config ───────────────────────────────────────────────────────────────────
BASE_MODEL      = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
MAX_SEQ_LENGTH  = 4096   # Reduce to 2048 if you get OOM on T4
LOAD_IN_4BIT    = True
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05
TARGET_MODULES  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                   'gate_proj', 'up_proj', 'down_proj']
# ─────────────────────────────────────────────────────────────────────────────

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect bf16/fp16
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',  # saves VRAM
    random_state=42,
)

print(model.print_trainable_parameters())

In [ ]:
# @title Step 5 — Load and format the dataset
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                obj = json.loads(line)
                if isinstance(obj, dict) and 'messages' in obj:
                    rows.append(obj)
    return rows

train_rows = load_jsonl(TRAIN_PATH)
val_rows   = load_jsonl(VAL_PATH) if os.path.exists(VAL_PATH) else []

print(f'Train examples: {len(train_rows)}')
print(f'Val examples:   {len(val_rows)}')

if len(train_rows) == 0:
    raise RuntimeError(f'No training data found at {TRAIN_PATH}. Upload your train.jsonl to Drive.')

# Apply Llama 3.1 chat template
def format_examples(examples):
    texts = []
    for messages in examples['messages']:
        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        )
    return {'text': texts}

train_ds = Dataset.from_list(train_rows).map(format_examples, batched=True, remove_columns=['messages'])
val_ds   = Dataset.from_list(val_rows).map(format_examples, batched=True, remove_columns=['messages']) if val_rows else None

print(f'\nSample formatted text (first 300 chars):')
print(train_ds[0]['text'][:300])

In [ ]:
# @title Step 6 — Train
from trl import SFTTrainer, SFTConfig
import math

# ── Training hyperparameters ─────────────────────────────────────────────────
EPOCHS        = 3    # was 2 -- raised now that loss/val-loss showed a healthy
                      # decreasing trend rather than overfitting; load_best_model_at_end
                      # guards against a worse 3rd-epoch checkpoint being selected anyway.
BATCH_SIZE    = 2    # reduce to 1 if OOM
GRAD_ACCUM    = 4
LEARNING_RATE = 2e-4
# ─────────────────────────────────────────────────────────────────────────────

# Warmup as a fraction of total optimizer steps, not a hardcoded step count --
# a fixed warmup_steps=10 was ~42% of training on a 92-example/2-epoch run
# (only 24 total steps), which is far more warmup than the usual 3-10% and
# likely under-uses the peak learning rate. Recomputed here from the actual
# dataset size so it stays proportionate as the dataset grows.
WARMUP_RATIO = 0.06
_steps_per_epoch = max(1, math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM)))
_total_steps = _steps_per_epoch * EPOCHS
WARMUP_STEPS = max(5, round(_total_steps * WARMUP_RATIO))
print(f'Steps/epoch={_steps_per_epoch}  Total steps={_total_steps}  Warmup steps={WARMUP_STEPS} ({WARMUP_STEPS/_total_steps:.1%})')

use_bf16 = torch.cuda.is_bf16_supported()
print(f'Using {"bf16" if use_bf16 else "fp16"}')

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=False,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir='/content/checkpoints',
        save_strategy='epoch',
        eval_strategy='epoch' if val_ds else 'no',
        load_best_model_at_end=bool(val_ds),
        report_to='none',
    ),
)

trainer_stats = trainer.train()
print(f'\nTraining complete. Loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# @title Step 7 — Save LoRA adapter to Google Drive
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to {OUTPUT_DIR}')

# List saved files
import os
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1e6
    print(f'  {f}  ({size:.1f} MB)')

---

## Resuming in a later session?

If you already trained and saved an adapter to Drive in a **previous** Colab
session, and just want to reload it (e.g. to re-run Step 8b/8c, or to re-run
Step 9's GGUF export) without retraining from scratch, use the two
**QUICK START** cells below **instead of** Steps 0-7 above.

**First time training, or training on new/updated data? Skip QUICK START
entirely** -- you already just ran Steps 0-7 above, so continue straight to
**Step 8b** below the QUICK START cells.


In [ ]:
# @title QUICK START (1/2) -- Install Unsloth + Restart Runtime
# Use this ONLY when resuming after a session restart, with a model already
# trained and saved on Drive from a previous run. If you have not trained yet,
# skip this entirely and start at 'Step 0 -- Check GPU' instead.
#
# After this cell restarts the runtime, run the NEXT cell
# ('QUICK START (2/2)') -- do not re-run this one.
import subprocess, importlib.metadata, os

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print('OK:', cmd[:60]) if r.returncode == 0 else print('ERR:', r.stderr[-300:])

print('Installing unsloth (takes ~3 min)...')
run('pip install unsloth --quiet')
run('pip install --force-reinstall --no-cache-dir unsloth --quiet')

# Deliberately do NOT `import unsloth` in this same cell/kernel session --
# see the identical note in Step 1 for why (numpy mid-session-upgrade crash).
print('Installed unsloth version:', importlib.metadata.version('unsloth'))
print('RESTARTING THE COLAB RUNTIME NOW -- this is expected.')
print('After it restarts, run the NEXT cell (QUICK START 2/2).')
os.kill(os.getpid(), 9)

In [ ]:
# @title QUICK START (2/2) -- Load the fine-tuned model from Drive
# Run this AFTER QUICK START (1/2) has restarted the runtime.
from google.colab import drive, userdata
from huggingface_hub import login
from unsloth import FastLanguageModel
import torch, os

print('Mounting Drive...')
drive.mount('/content/drive')

print('Logging into HuggingFace...')
login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)

OUTPUT_DIR    = '/content/drive/MyDrive/communiq/lora_output'
MAX_SEQ_LENGTH = 4096
DATA_DIR      = '/content/drive/MyDrive/communiq/data'
TRAIN_PATH    = f'{DATA_DIR}/train.jsonl'
VAL_PATH      = f'{DATA_DIR}/val.jsonl'

print('Loading fine-tuned model from Drive...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

from IPython.display import display, HTML
display(HTML('''
<div style="font-family:'Segoe UI',sans-serif;background:#052e16;border-radius:10px;padding:16px 20px;border:1px solid #166534;margin-top:8px;">
  <span style="color:#4ade80;font-size:15px;font-weight:600;">Fine-tuned model loaded from Drive -- no retraining needed!</span><br>
  <span style="color:#86efac;font-size:13px;">Now scroll down and run <b>Step 8b</b> (fine-tuned responses) and <b>Step 8c</b> (base vs fine-tuned comparison).</span>
</div>
'''))

In [ ]:
# @title Step 8b — Fine-tuned Model Inference (run this first)
import torch
from unsloth import FastLanguageModel
from IPython.display import display, HTML
import time

FastLanguageModel.for_inference(model)

TEST_SCENARIOS = [
    {
        "icon": "🎯",
        "type": "HR Interview",
        "label": "Weak, filler-heavy candidate",
        "tag_color": "#4f46e5",
        "messages": [
            {"role": "system",    "content": "You are a senior HR interviewer at a tech company. Ask one behavioural question at a time. Keep replies to 2 sentences. Be warm but professional."},
            {"role": "assistant", "content": "Tell me about yourself and what brought you to apply for this role."},
            {"role": "user",      "content": "Um, so basically I am, like, a CS student and I guess I enjoy coding you know, I have done some projects and stuff, hopefully that counts."},
        ],
    },
    {
        "icon": "💼",
        "type": "Sales Pitch",
        "label": "Rambling candidate, exec losing patience",
        "tag_color": "#0891b2",
        "messages": [
            {"role": "system",    "content": "You are a time-pressed C-suite executive. You reward brevity and clarity, lose patience with rambling. Keep replies to 1-2 sentences."},
            {"role": "assistant", "content": "You have thirty seconds — why should I care about your product?"},
            {"role": "user",      "content": "So first I want to thank you for the time and, you know, our journey started when my cofounder and I were in college, and we thought about data, and then we pivoted twice, anyway the product does analytics and reporting and dashboards and also some AI maybe."},
        ],
    },
    {
        "icon": "🔥",
        "type": "Client Handling",
        "label": "Angry client — candidate recovers well",
        "tag_color": "#dc2626",
        "messages": [
            {"role": "system",    "content": "You are an upset enterprise client whose product delivery was delayed 3 weeks causing financial loss. Demand answers. Calm down if the candidate acknowledges and offers concrete steps."},
            {"role": "assistant", "content": "This delay cost us our launch window. I need answers and I need them now."},
            {"role": "user",      "content": "You are absolutely right to be upset and I take full ownership. The delay was caused by a supplier slip that hit our QA gate. Here is what we are doing: expedited shipping at our cost, a dedicated support line for your team for 60 days, and a 10 percent credit on your next order. You will have written confirmation by end of day."},
        ],
    },
]

def run_inference(messages, max_new_tokens=150):
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    with torch.no_grad():
        output = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, top_p=0.9, repetition_penalty=1.1, use_cache=True,
        )
    return tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()

# ── Collect responses ──
finetuned_responses = []
display(HTML('''
<div style="font-family:'Segoe UI',sans-serif;background:#0f172a;border-radius:12px;padding:20px 24px;margin:8px 0;color:#94a3b8;font-size:13px;">
  ⏳ &nbsp;Running fine-tuned model inference on 3 scenarios...
</div>'''))

for s in TEST_SCENARIOS:
    t0 = time.time()
    resp = run_inference(s['messages'])
    elapsed = time.time() - t0
    finetuned_responses.append({'response': resp, 'elapsed': elapsed})

# ── Render results ──
cards_html = ''
for i, (s, r) in enumerate(zip(TEST_SCENARIOS, finetuned_responses)):
    user_text = s['messages'][-1]['content']
    ai_text   = s['messages'][-2]['content']
    cards_html += f'''
    <div style="background:#1e293b;border-radius:12px;padding:22px 26px;margin-bottom:18px;border-left:4px solid {s["tag_color"]};">
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:14px;">
        <span style="font-size:20px;">{s["icon"]}</span>
        <div>
          <span style="background:{s["tag_color"]};color:#fff;font-size:11px;font-weight:700;padding:3px 10px;border-radius:20px;letter-spacing:0.5px;">{s["type"].upper()}</span>
          <span style="color:#94a3b8;font-size:12px;margin-left:8px;">{s["label"]}</span>
        </div>
      </div>

      <div style="margin-bottom:10px;">
        <div style="color:#64748b;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:1px;margin-bottom:6px;">AI Opens</div>
        <div style="background:#0f172a;border-radius:8px;padding:10px 14px;color:#e2e8f0;font-size:13.5px;line-height:1.6;border:1px solid #334155;">
          {ai_text}
        </div>
      </div>

      <div style="margin-bottom:14px;">
        <div style="color:#64748b;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:1px;margin-bottom:6px;">Candidate Responds</div>
        <div style="background:#0f172a;border-radius:8px;padding:10px 14px;color:#fbbf24;font-size:13px;line-height:1.6;border:1px solid #334155;font-style:italic;">
          "{user_text}"
        </div>
      </div>

      <div>
        <div style="color:#64748b;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:1px;margin-bottom:6px;">
          🤖 Fine-tuned CommuniQ Model
          <span style="color:#22c55e;font-weight:400;text-transform:none;font-size:11px;margin-left:8px;">✓ {r["elapsed"]:.1f}s</span>
        </div>
        <div style="background:#052e16;border-radius:8px;padding:12px 16px;color:#86efac;font-size:14px;line-height:1.7;border:1px solid #166534;">
          {r["response"]}
        </div>
      </div>
    </div>'''

display(HTML(f'''
<div style="font-family:'Segoe UI',sans-serif;max-width:860px;margin:0 auto;">
  <div style="background:linear-gradient(135deg,#1e1b4b,#0f172a);border-radius:14px;padding:24px 28px;margin-bottom:20px;">
    <div style="display:flex;align-items:center;gap:12px;margin-bottom:6px;">
      <span style="font-size:26px;">🧠</span>
      <div>
        <div style="color:#e2e8f0;font-size:20px;font-weight:700;letter-spacing:-0.3px;">CommuniQ Fine-tuned Model</div>
        <div style="color:#6366f1;font-size:13px;font-weight:500;">meta-llama/Meta-Llama-3.1-8B-Instruct + QLoRA · {len(train_rows)} training examples · {EPOCHS} epochs</div>
      </div>
    </div>
    <div style="color:#94a3b8;font-size:12.5px;margin-top:10px;">
      The model below was fine-tuned specifically on CommuniQ conversation data — HR interviews, sales pitches, client handling, GDs and technical rounds.
      Compare its responses against the base model in the next cell.
    </div>
  </div>
  {cards_html}
  <div style="background:#0f172a;border-radius:10px;padding:14px 20px;color:#22c55e;font-size:13px;border:1px solid #166534;">
    ✅ Fine-tuned inference complete — run the next cell to see base model responses on the same prompts.
  </div>
</div>
'''))


In [ ]:
# @title Step 8c — Base Model Responses (same prompts, no fine-tuning)
import time
from IPython.display import display, HTML

display(HTML('''
<div style="font-family:'Segoe UI',sans-serif;background:#0f172a;border-radius:12px;padding:20px 24px;margin:8px 0;color:#94a3b8;font-size:13px;">
  ⏳ &nbsp;Loading base Llama 3.1 8B (no fine-tuning)... this may take a minute.
</div>'''))

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name='meta-llama/Meta-Llama-3.1-8B-Instruct',
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(base_model)

def run_base_inference(messages, max_new_tokens=150):
    inputs = base_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    with torch.no_grad():
        output = base_model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, top_p=0.9, repetition_penalty=1.1, use_cache=True,
        )
    return base_tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()

# ── Collect base responses ──
base_responses = []
for s in TEST_SCENARIOS:
    t0 = time.time()
    resp = run_base_inference(s['messages'])
    base_responses.append({'response': resp, 'elapsed': time.time() - t0})

# ── Render side-by-side comparison ──
cards_html = ''
for i, (s, ft, base) in enumerate(zip(TEST_SCENARIOS, finetuned_responses, base_responses)):
    user_text = s['messages'][-1]['content']
    ai_text   = s['messages'][-2]['content']
    cards_html += f'''
    <div style="background:#1e293b;border-radius:12px;padding:22px 26px;margin-bottom:20px;border-left:4px solid {s["tag_color"]};">
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:16px;">
        <span style="font-size:20px;">{s["icon"]}</span>
        <div>
          <span style="background:{s["tag_color"]};color:#fff;font-size:11px;font-weight:700;padding:3px 10px;border-radius:20px;">{s["type"].upper()}</span>
          <span style="color:#94a3b8;font-size:12px;margin-left:8px;">{s["label"]}</span>
        </div>
      </div>

      <div style="margin-bottom:12px;">
        <div style="color:#64748b;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:1px;margin-bottom:5px;">Candidate Said</div>
        <div style="background:#0f172a;border-radius:8px;padding:10px 14px;color:#fbbf24;font-size:13px;font-style:italic;line-height:1.6;border:1px solid #334155;">
          "{user_text}"
        </div>
      </div>

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-top:4px;">
        <div>
          <div style="color:#64748b;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:1px;margin-bottom:6px;">
            ❌ Base Llama 3.1 8B
            <span style="color:#ef4444;font-weight:400;text-transform:none;font-size:11px;margin-left:6px;">{base["elapsed"]:.1f}s</span>
          </div>
          <div style="background:#2d1515;border-radius:8px;padding:12px 14px;color:#fca5a5;font-size:13px;line-height:1.7;border:1px solid #7f1d1d;min-height:90px;">
            {base["response"]}
          </div>
        </div>
        <div>
          <div style="color:#64748b;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:1px;margin-bottom:6px;">
            ✅ CommuniQ Fine-tuned
            <span style="color:#22c55e;font-weight:400;text-transform:none;font-size:11px;margin-left:6px;">{ft["elapsed"]:.1f}s</span>
          </div>
          <div style="background:#052e16;border-radius:8px;padding:12px 14px;color:#86efac;font-size:13px;line-height:1.7;border:1px solid #166534;min-height:90px;">
            {ft["response"]}
          </div>
        </div>
      </div>
    </div>'''

display(HTML(f'''
<div style="font-family:'Segoe UI',sans-serif;max-width:960px;margin:0 auto;">

  <div style="background:linear-gradient(135deg,#1a1a2e,#16213e);border-radius:14px;padding:24px 28px;margin-bottom:20px;">
    <div style="color:#e2e8f0;font-size:20px;font-weight:700;margin-bottom:6px;">⚖️ Base vs Fine-tuned — Side by Side</div>
    <div style="color:#94a3b8;font-size:13px;">Same scenario. Same prompt. Same temperature. Different training.</div>
    <div style="display:flex;gap:24px;margin-top:14px;">
      <div style="display:flex;align-items:center;gap:8px;">
        <div style="width:12px;height:12px;border-radius:3px;background:#7f1d1d;"></div>
        <span style="color:#94a3b8;font-size:12px;">Base Llama 3.1 8B — no domain training</span>
      </div>
      <div style="display:flex;align-items:center;gap:8px;">
        <div style="width:12px;height:12px;border-radius:3px;background:#166534;"></div>
        <span style="color:#94a3b8;font-size:12px;">CommuniQ fine-tuned — {len(train_rows)} examples, QLoRA, {EPOCHS} epochs</span>
      </div>
    </div>
  </div>

  {cards_html}

  <div style="background:#0f172a;border-radius:12px;padding:20px 24px;border:1px solid #1e293b;margin-top:4px;">
    <div style="color:#e2e8f0;font-size:14px;font-weight:600;margin-bottom:12px;">🔍 What to look for</div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div>
        <div style="color:#f87171;font-size:12px;font-weight:600;margin-bottom:6px;">Base model tends to:</div>
        <ul style="color:#94a3b8;font-size:12.5px;line-height:1.9;margin:0;padding-left:16px;">
          <li>Give generic, assistant-style replies</li>
          <li>Over-explain or add unnecessary advice</li>
          <li>Break character by being too helpful</li>
          <li>Miss the specific tone of the scenario</li>
        </ul>
      </div>
      <div>
        <div style="color:#4ade80;font-size:12px;font-weight:600;margin-bottom:6px;">Fine-tuned model tends to:</div>
        <ul style="color:#94a3b8;font-size:12.5px;line-height:1.9;margin:0;padding-left:16px;">
          <li>Stay in persona (interviewer / exec / client)</li>
          <li>Probe with specific follow-up questions</li>
          <li>React appropriately to candidate quality</li>
          <li>Match CommuniQ's concise conversational tone</li>
        </ul>
      </div>
    </div>
  </div>
</div>
'''))


## ⚠️ A note before Steps 8d-8f

The three cells below (8d, 8e, 8f) try to test this fine-tune as a **JSON-rubric session evaluator** (scoring a transcript on fluency/confidence/vocabulary/etc. and returning structured JSON).

**This is not what this model is trained for, and its result here doesn't reflect the fine-tune's real quality.** The training data (all 92-180+ examples) is 100% roleplay-turn generation -- this model was never shown a single JSON-scoring example. In production, evaluation runs on a completely separate backend (`EVAL_LLM_BACKEND`, typically Groq) via `ai-server/app/api/v1/evaluate.py` -- this fine-tune's only job is `LLM_BACKEND` (generating the AI's conversational replies during a roleplay session).

**The check that actually matters is Step 8b/8c above** -- the qualitative base-vs-fine-tuned comparison on the model's real job (roleplay reply generation).

Keep 8d-8f only if you're curious or plan to separately train this model for evaluation too (which would need its own JSON-labeled training data); otherwise, skip straight to Step 9.

In [ ]:
# @title Step 8d (OPTIONAL / OUT-OF-SCOPE) -- Fine-tuned Model as a JSON-rubric evaluator
# Session 1: Strong candidate selling a mobile phone (has some mispronunciations/fillers but overall good)
# Session 2: Weak candidate on a job interview roleplay
# After each session, the fine-tuned model evaluates the candidate.

from IPython.display import display, HTML
import torch, json, time
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# ── Two pre-written realistic sessions ────────────────────────────────────────
SESSIONS = [
    {
        "id": "session_1",
        "icon": "📱",
        "scenario": "Sales Roleplay — Selling a Mobile Phone",
        "candidate_label": "Strong Candidate",
        "candidate_color": "#22c55e",
        "border_color": "#166534",
        "bg_color": "#052e16",
        "transcript": [
            {"role": "user",      "content": "Good morning! I am Rahul from NovaTech. I wanted to, um, show you our latest flagship device, the NovaX Pro. It is, I believe, one of the most, uh, innov-ative smartphones in its price segment right now."},
            {"role": "assistant", "content": "Good morning. I already have a phone I am happy with. Why should I consider switching?"},
            {"role": "user",      "content": "That is a great question. So the NovaX Pro has a 200 megapixel camera with AI-enhanced night mode, which means your photos in low light will be — how do I put it — exceptionally clear. The battery is five thousand milliamp-hour, which in normal usage gives you easily two days without charging. The processer is also the latest Snapdragon 8 Gen 3."},
            {"role": "assistant", "content": "Sounds similar to other flagships. What makes it different from, say, a Samsung or iPhone at the same price?"},
            {"role": "user",      "content": "The key differensiation — sorry, differentiation — is our after-sales support. We offer three years of free screen replacement, which no other brand does at this price point of thirty-five thousand rupees. Also our bloat-ware is zero, it runs stock Android which most techie customers really appre-ciate."},
            {"role": "assistant", "content": "Interesting. What about software updates? My current phone stopped getting updates after two years."},
            {"role": "user",      "content": "We guarantee five years of Android up-dates and security patches. That is directly from our CEO's commitment last quarter. If you are buying a device at this price, you want longe-vity, and NovaX gives you that assurance in writing with the box itself."},
            {"role": "assistant", "content": "Alright, you have made some compelling points. I would like to think about it. Can you leave me a brochure?"},
            {"role": "user",      "content": "Absolutely. Here is our brochure and my visiting card. I will also send you a what-sapp message with a QR code for our EMI options — zero percent for twelve months if you decide this week. Thank you for your time, it was a pleasure meeting you."},
        ],
    },
    {
        "id": "session_2",
        "icon": "💼",
        "scenario": "Job Interview Roleplay — Software Developer Position",
        "candidate_label": "Weak Candidate",
        "candidate_color": "#ef4444",
        "border_color": "#7f1d1d",
        "bg_color": "#2d1515",
        "transcript": [
            {"role": "user",      "content": "Um, hi, good morning, I am Priya, I am here for the, uh, softwear developer role, I think."},
            {"role": "assistant", "content": "Good morning Priya. Please tell me about yourself and your experience with software development."},
            {"role": "user",      "content": "So basically I am like a fresher, I did my B-Tech last year, I have done some projects, like one was a, um, website? It was using HTML and CSS and maybe some Java-script, I am not fully sure, my friend helped me a lot with it actually."},
            {"role": "assistant", "content": "Can you describe a technical challenge you faced in that project and how you solved it?"},
            {"role": "user",      "content": "Hmm, so there was like a bug, I think, where the page was not loading properly or something, and I just, like, googled it and then copy-pasted some code from Stack Overflow and it worked, so yeah that is how I fixed it basically."},
            {"role": "assistant", "content": "What programming languages are you most comfortable with?"},
            {"role": "user",      "content": "I know Python a little bit, and Java, and also C maybe, I studied them in college. I have not really used them after college too much to be honest. I am a fast learner though, I think I can pick up whatever is needed."},
            {"role": "assistant", "content": "Where do you see yourself in five years?"},
            {"role": "user",      "content": "Um, I want to grow, like, professionally and maybe become a senior developer or something. I am not fully sure yet what exactly I want but I know I want to do something good in the tech field, hopefully."},
        ],
    },
]

EVAL_SYSTEM = """You are CommuniQ's expert communication evaluator. Evaluate the CANDIDATE's communication only.

You MUST respond with ONLY a raw JSON object. No markdown, no code fences, no explanation before or after. Start your response with { and end with }.

Use exactly this structure:
{"scores":{"fluency":{"score":7,"comment":"one sentence"},"confidence":{"score":6,"comment":"one sentence"},"vocabulary":{"score":5,"comment":"one sentence"},"clarity":{"score":7,"comment":"one sentence"},"domain_knowledge":{"score":6,"comment":"one sentence"}},"overall_score":6,"overall_summary":"2-3 sentence summary","mispronounced_words":[{"said":"innov-ative","correct":"innovative","note":"word split incorrectly"}],"filler_words_detected":["um","like","basically"],"vocabulary_issues":["weak phrase example"],"strengths":["strength 1","strength 2"],"areas_to_improve":["area 1","area 2"]}

Replace all values with your actual evaluation. Scores are integers 1-10. Be honest and specific."""

def build_eval_prompt(session):
    lines = []
    for m in session['transcript']:
        role = 'CANDIDATE (Employee)' if m['role'] == 'user' else 'AI CLIENT/INTERVIEWER'
        lines.append(f"{role}: {m['content']}")
    transcript_text = "\n\n".join(lines)
    return f"""Scenario: {session['scenario']}
Candidate level: {session['candidate_label']}

FULL TRANSCRIPT:
{transcript_text}

Now evaluate the CANDIDATE only (not the AI). Be specific about mispronounced words you detected in their speech."""

def run_inference(messages, max_new_tokens=600, temperature=0.2):
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.9, repetition_penalty=1.05, use_cache=True,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

def parse_eval(raw):
    # Try direct parse
    try:
        s = raw.find('{'); e = raw.rfind('}') + 1
        if s >= 0 and e > s:
            return json.loads(raw[s:e])
    except: pass
    # Fallback: build a stub so the UI still renders with the raw text
    return {
        "scores": {
            "fluency":         {"score": 0, "comment": "Parse error — see raw output"},
            "confidence":      {"score": 0, "comment": ""},
            "vocabulary":      {"score": 0, "comment": ""},
            "clarity":         {"score": 0, "comment": ""},
            "domain_knowledge":{"score": 0, "comment": ""},
        },
        "overall_score": 0,
        "overall_summary": f"Raw model output: {raw[:400]}",
        "mispronounced_words": [],
        "filler_words_detected": [],
        "vocabulary_issues": [],
        "strengths": [],
        "areas_to_improve": ["Model did not return valid JSON — try increasing max_new_tokens or temperature"],
    }

def score_color(s):
    if s >= 8: return '#22c55e'
    if s >= 6: return '#f59e0b'
    if s >= 4: return '#f97316'
    return '#ef4444'

def score_bar_html(score):
    c = score_color(score)
    return f'''<div style="display:flex;align-items:center;gap:8px;">
      <div style="flex:1;background:#0f172a;border-radius:4px;height:7px;overflow:hidden;">
        <div style="width:{score*10}%;background:{c};height:100%;border-radius:4px;"></div>
      </div>
      <span style="color:{c};font-weight:700;font-size:12px;min-width:28px;">{score}/10</span>
    </div>'''

def render_session(session, eval_data, elapsed, model_label, header_gradient):
    # Always render transcript
    tr_html = ''
    for m in session['transcript']:
        is_user = m['role'] == 'user'
        label  = '🧑 Candidate' if is_user else '🤖 AI Client / Interviewer'
        lcolor = '#fbbf24' if is_user else '#818cf8'
        bgcol  = '#1a1a2e' if is_user else '#0f172a'
        tr_html += f'''<div style="display:flex;gap:10px;margin-bottom:10px;">
          <div style="min-width:160px;color:{lcolor};font-size:11px;font-weight:700;padding-top:2px;">{label}</div>
          <div style="background:{bgcol};border-radius:8px;padding:10px 14px;color:#e2e8f0;font-size:13px;line-height:1.65;flex:1;border:1px solid #334155;">{m["content"]}</div>
        </div>'''

    if not eval_data:
        eval_data = parse_eval("")  # get stub

    sc = eval_data.get('scores', {})
    overall = eval_data.get('overall_score', 0)
    oc = score_color(overall)

    dims_html = ''
    for dim in ['fluency','confidence','vocabulary','clarity','domain_knowledge']:
        if dim not in sc: continue
        d = sc[dim]
        label = dim.replace('_',' ').title()
        dims_html += f'''<div style="margin-bottom:12px;">
          <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
            <span style="color:#94a3b8;font-size:12px;font-weight:600;">{label}</span>
          </div>
          {score_bar_html(d["score"])}
          <div style="color:#64748b;font-size:11.5px;margin-top:4px;line-height:1.5;">{d.get("comment","")}</div>
        </div>'''

    def pill_list(items, bg, color):
        if not items: return '<span style="color:#475569;font-size:12px;">None detected</span>'
        pills = ''.join(f'<span style="background:{bg};color:{color};padding:3px 10px;border-radius:20px;font-size:12px;margin:3px 2px;display:inline-block;">{i}</span>' for i in items)
        return f'<div style="margin-top:4px;">{pills}</div>'

    mispr_html = ''
    for w in eval_data.get('mispronounced_words', []):
        mispr_html += f'''<div style="background:#1e293b;border-radius:8px;padding:8px 12px;margin-bottom:6px;border-left:3px solid #ef4444;">
          <span style="color:#f87171;font-weight:700;font-size:13px;">"{w.get("said","")}"</span>
          <span style="color:#64748b;font-size:12px;"> → correct: </span>
          <span style="color:#4ade80;font-weight:600;font-size:13px;">"{w.get("correct","")}"</span>
          <div style="color:#94a3b8;font-size:11.5px;margin-top:3px;">{w.get("note","")}</div>
        </div>'''

    strengths_html = ''.join(f'<div style="color:#86efac;font-size:12.5px;margin-bottom:4px;">✓ {s}</div>' for s in eval_data.get('strengths',[]))
    improve_html   = ''.join(f'<div style="color:#fca5a5;font-size:12.5px;margin-bottom:4px;">✗ {s}</div>' for s in eval_data.get('areas_to_improve',[]))

    return f'''
<div style="background:#1e293b;border-radius:14px;margin-bottom:24px;overflow:hidden;border:1px solid #334155;">
  <div style="background:{header_gradient};padding:18px 24px;">
    <div style="display:flex;align-items:center;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:10px;">
        <span style="font-size:22px;">{session["icon"]}</span>
        <div>
          <div style="color:#fff;font-size:16px;font-weight:700;">{session["scenario"]}</div>
          <div style="display:flex;align-items:center;gap:8px;margin-top:3px;">
            <span style="background:rgba(255,255,255,0.15);color:#fff;font-size:11px;padding:2px 8px;border-radius:12px;">{session["candidate_label"]}</span>
            <span style="color:rgba(255,255,255,0.6);font-size:11px;">{model_label} · {elapsed:.1f}s</span>
          </div>
        </div>
      </div>
      <div style="text-align:center;">
        <div style="font-size:38px;font-weight:800;color:{oc};line-height:1;">{overall}</div>
        <div style="color:rgba(255,255,255,0.5);font-size:10px;">/ 10</div>
      </div>
    </div>
  </div>

  <div style="padding:20px 24px;">
    <div style="color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:12px;">Full Conversation</div>
    <div style="background:#0a0f1e;border-radius:10px;padding:16px;margin-bottom:20px;border:1px solid #1e293b;">{tr_html}</div>

    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:20px;">
      <div>
        <div style="color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:12px;">Score Breakdown</div>
        {dims_html}
      </div>
      <div>
        <div style="color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:10px;">Mispronounced Words</div>
        {mispr_html if mispr_html else '<span style="color:#475569;font-size:12px;">None detected</span>'}
        <div style="margin-top:16px;">
          <div style="color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:8px;">Filler Words Detected</div>
          {pill_list(eval_data.get("filler_words_detected",[]), "#1e293b", "#f59e0b")}
        </div>
        <div style="margin-top:12px;">
          <div style="color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:8px;">Vocabulary Issues</div>
          {pill_list(eval_data.get("vocabulary_issues",[]), "#1e293b", "#f87171")}
        </div>
      </div>
    </div>

    <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div style="background:#0f2a1a;border-radius:8px;padding:14px 16px;border:1px solid #166534;">
        <div style="color:#4ade80;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:8px;">Strengths</div>
        {strengths_html or '<div style="color:#475569;font-size:12px;">—</div>'}
      </div>
      <div style="background:#2d1515;border-radius:8px;padding:14px 16px;border:1px solid #7f1d1d;">
        <div style="color:#f87171;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:8px;">Areas to Improve</div>
        {improve_html or '<div style="color:#475569;font-size:12px;">—</div>'}
      </div>
    </div>

    <div style="background:#0f172a;border-radius:8px;padding:14px 16px;margin-top:16px;border:1px solid #1e293b;">
      <div style="color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:6px;">Overall Summary</div>
      <div style="color:#cbd5e1;font-size:13px;line-height:1.7;">{eval_data.get("overall_summary","")}</div>
    </div>
  </div>
</div>'''

# ── Run evaluations ──
display(HTML('''<div style="font-family:'Segoe UI',sans-serif;background:#0f172a;border-radius:10px;padding:16px 20px;color:#94a3b8;font-size:13px;margin-bottom:12px;">
  ⏳ Fine-tuned model evaluating 2 sessions... (may take 1-2 min)
</div>'''))

ft_results = []
for s in SESSIONS:
    msgs = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": build_eval_prompt(s)},
    ]
    t0 = time.time()
    raw = run_inference(msgs, max_new_tokens=700, temperature=0.2)
    ft_results.append({'parsed': parse_eval(raw), 'raw': raw, 'elapsed': time.time()-t0})

# ── Render ──
gradients = [
    'linear-gradient(135deg,#065f46,#0f172a)',
    'linear-gradient(135deg,#7f1d1d,#0f172a)',
]
all_cards = ''.join(
    render_session(s, r['parsed'], r['elapsed'], '🧠 Fine-tuned CommuniQ', g)
    for s, r, g in zip(SESSIONS, ft_results, gradients)
)

display(HTML(f'''
<div style="font-family:'Segoe UI',sans-serif;max-width:980px;margin:0 auto;">
  <div style="background:linear-gradient(135deg,#1e1b4b,#0f172a);border-radius:14px;padding:22px 28px;margin-bottom:22px;">
    <div style="display:flex;align-items:center;gap:12px;">
      <span style="font-size:26px;">🧠</span>
      <div>
        <div style="color:#e2e8f0;font-size:19px;font-weight:700;">Fine-tuned CommuniQ Model — Session Evaluation</div>
        <div style="color:#6366f1;font-size:12.5px;margin-top:2px;">2 roleplay sessions · Full transcript · Mispronunciation + filler detection · Score breakdown</div>
      </div>
    </div>
  </div>
  {all_cards}
  <div style="background:#052e16;border-radius:10px;padding:14px 20px;color:#22c55e;font-size:13px;border:1px solid #166534;">
    ✅ Fine-tuned model evaluation complete. Run the next cell for base model comparison.
  </div>
</div>
'''))

# Store for comparison cell
_ft_results = ft_results

In [ ]:
# @title Step 8e (OPTIONAL / OUT-OF-SCOPE) -- Base Model as a JSON-rubric evaluator
from IPython.display import display, HTML
import time

display(HTML('''<div style="font-family:'Segoe UI',sans-serif;background:#0f172a;border-radius:10px;padding:16px 20px;color:#94a3b8;font-size:13px;margin-bottom:12px;">
  ⏳ Loading base Llama 3.1 8B (no fine-tuning)...
</div>'''))

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name='meta-llama/Meta-Llama-3.1-8B-Instruct',
    max_seq_length=4096, load_in_4bit=True, dtype=None,
)
FastLanguageModel.for_inference(base_model)

def run_base(messages, max_new_tokens=700, temperature=0.2):
    inputs = base_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = base_model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.9, repetition_penalty=1.05, use_cache=True,
        )
    return base_tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

display(HTML('''<div style="font-family:'Segoe UI',sans-serif;background:#0f172a;border-radius:10px;padding:16px 20px;color:#94a3b8;font-size:13px;margin-bottom:12px;">
  ⏳ Base model evaluating 2 sessions...
</div>'''))

base_results = []
for s in SESSIONS:
    msgs = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": build_eval_prompt(s)},
    ]
    t0 = time.time()
    raw = run_base(msgs, max_new_tokens=700, temperature=0.2)
    base_results.append({'parsed': parse_eval(raw), 'raw': raw, 'elapsed': time.time()-t0})

gradients = [
    'linear-gradient(135deg,#1e3a5f,#0f172a)',
    'linear-gradient(135deg,#3b1f5e,#0f172a)',
]
all_cards = ''.join(
    render_session(s, r['parsed'], r['elapsed'], '⚙️ Base Llama 3.1 8B', g)
    for s, r, g in zip(SESSIONS, base_results, gradients)
)

display(HTML(f'''
<div style="font-family:'Segoe UI',sans-serif;max-width:980px;margin:0 auto;">
  <div style="background:linear-gradient(135deg,#1a1a2e,#0f172a);border-radius:14px;padding:22px 28px;margin-bottom:22px;">
    <div style="display:flex;align-items:center;gap:12px;">
      <span style="font-size:26px;">⚙️</span>
      <div>
        <div style="color:#e2e8f0;font-size:19px;font-weight:700;">Base Llama 3.1 8B — Same Sessions, No Fine-tuning</div>
        <div style="color:#6366f1;font-size:12.5px;margin-top:2px;">Identical prompts · No domain training · Compare with fine-tuned results above</div>
      </div>
    </div>
  </div>
  {all_cards}
  <div style="background:#0f172a;border-radius:10px;padding:14px 20px;color:#94a3b8;font-size:13px;border:1px solid #1e293b;">
    ✅ Base model evaluation complete. Run the next cell to see the side-by-side comparison.
  </div>
</div>
'''))

_base_results = base_results

In [ ]:
# @title Step 8f (OPTIONAL / OUT-OF-SCOPE) -- Head-to-Head Comparison (evaluator task)
from IPython.display import display, HTML

def get_score(result, dim):
    try: return result['parsed']['scores'][dim]['score']
    except: return 0

def get_overall(result):
    try: return result['parsed']['overall_score']
    except: return 0

def get_fillers(result):
    try: return result['parsed'].get('filler_words_detected', [])
    except: return []

def get_mispr(result):
    try: return result['parsed'].get('mispronounced_words', [])
    except: return []

def get_summary(result):
    try: return result['parsed'].get('overall_summary', 'N/A')
    except: return 'N/A'

def delta_badge(ft_score, base_score):
    diff = ft_score - base_score
    if diff > 0:   return f'<span style="color:#22c55e;font-size:11px;font-weight:700;">▲ +{diff}</span>'
    if diff < 0:   return f'<span style="color:#ef4444;font-size:11px;font-weight:700;">▼ {diff}</span>'
    return             f'<span style="color:#64748b;font-size:11px;">= 0</span>'

def compare_card(session, ft_r, base_r):
    dims = ['fluency','confidence','vocabulary','clarity','domain_knowledge']
    ft_overall   = get_overall(ft_r)
    base_overall = get_overall(base_r)
    ft_color     = score_color(ft_overall)
    base_color   = score_color(base_overall)
    winner       = 'Fine-tuned' if ft_overall >= base_overall else 'Base'
    winner_color = '#22c55e' if ft_overall >= base_overall else '#f59e0b'

    rows = ''
    for dim in dims:
        ft_s   = get_score(ft_r, dim)
        base_s = get_score(base_r, dim)
        label  = dim.replace('_',' ').title()
        rows += f'''<tr>
          <td style="color:#94a3b8;font-size:12.5px;padding:8px 12px;border-bottom:1px solid #1e293b;">{label}</td>
          <td style="text-align:center;padding:8px 12px;border-bottom:1px solid #1e293b;">
            <span style="color:{score_color(ft_s)};font-weight:700;font-size:13px;">{ft_s}</span>
          </td>
          <td style="text-align:center;padding:8px 12px;border-bottom:1px solid #1e293b;">
            <span style="color:{score_color(base_s)};font-weight:700;font-size:13px;">{base_s}</span>
          </td>
          <td style="text-align:center;padding:8px 12px;border-bottom:1px solid #1e293b;">{delta_badge(ft_s, base_s)}</td>
        </tr>'''

    # Filler & mispronunciation comparison
    ft_fillers   = get_fillers(ft_r)
    base_fillers = get_fillers(base_r)
    ft_mispr     = get_mispr(ft_r)
    base_mispr   = get_mispr(base_r)

    def tag(items, color):
        if not items: return '<span style="color:#475569;font-size:11px;">None</span>'
        return ' '.join(f'<span style="background:#1e293b;color:{color};padding:2px 8px;border-radius:12px;font-size:11px;">{i}</span>' for i in items[:8])

    def mispr_compact(items):
        if not items: return '<span style="color:#475569;font-size:11px;">None detected</span>'
        return '<br>'.join(f'<span style="color:#f87171;">"{w.get("said","")}"</span> → <span style="color:#4ade80;">"{w.get("correct","")}"</span>' for w in items[:5])

    return f'''
<div style="background:#1e293b;border-radius:14px;margin-bottom:24px;overflow:hidden;border:1px solid #334155;">
  <div style="background:linear-gradient(135deg,#1e1b4b,#0f172a);padding:16px 24px;display:flex;align-items:center;justify-content:space-between;">
    <div style="display:flex;align-items:center;gap:10px;">
      <span style="font-size:20px;">{session["icon"]}</span>
      <div>
        <div style="color:#e2e8f0;font-size:15px;font-weight:700;">{session["scenario"]}</div>
        <span style="color:{session["candidate_color"]};font-size:12px;font-weight:600;">{session["candidate_label"]}</span>
      </div>
    </div>
    <div style="background:rgba(255,255,255,0.08);border-radius:8px;padding:8px 14px;text-align:center;">
      <div style="color:#94a3b8;font-size:10px;text-transform:uppercase;letter-spacing:1px;">Better evaluator</div>
      <div style="color:{winner_color};font-weight:700;font-size:14px;margin-top:2px;">{winner} Model</div>
    </div>
  </div>

  <div style="padding:20px 24px;">
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-bottom:20px;">
      <div style="background:#052e16;border-radius:10px;padding:16px;text-align:center;border:1px solid #166534;">
        <div style="color:#4ade80;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;">🧠 Fine-tuned Overall</div>
        <div style="font-size:44px;font-weight:800;color:{ft_color};line-height:1.2;margin-top:4px;">{ft_overall}</div>
        <div style="color:#64748b;font-size:10px;">/10</div>
        <div style="color:#86efac;font-size:11.5px;margin-top:8px;line-height:1.5;">{get_summary(ft_r)[:120]}...</div>
      </div>
      <div style="background:#1a1a2e;border-radius:10px;padding:16px;text-align:center;border:1px solid #334155;">
        <div style="color:#94a3b8;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;">⚙️ Base Model Overall</div>
        <div style="font-size:44px;font-weight:800;color:{base_color};line-height:1.2;margin-top:4px;">{base_overall}</div>
        <div style="color:#64748b;font-size:10px;">/10</div>
        <div style="color:#94a3b8;font-size:11.5px;margin-top:8px;line-height:1.5;">{get_summary(base_r)[:120]}...</div>
      </div>
    </div>

    <table style="width:100%;border-collapse:collapse;background:#0f172a;border-radius:10px;overflow:hidden;margin-bottom:20px;">
      <thead>
        <tr style="background:#1e293b;">
          <th style="text-align:left;padding:10px 12px;color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;">Dimension</th>
          <th style="text-align:center;padding:10px 12px;color:#4ade80;font-size:11px;font-weight:700;text-transform:uppercase;">🧠 Fine-tuned</th>
          <th style="text-align:center;padding:10px 12px;color:#94a3b8;font-size:11px;font-weight:700;text-transform:uppercase;">⚙️ Base</th>
          <th style="text-align:center;padding:10px 12px;color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;">Delta</th>
        </tr>
      </thead>
      <tbody>{rows}</tbody>
    </table>

    <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div style="background:#0f172a;border-radius:10px;padding:14px 16px;border:1px solid #1e293b;">
        <div style="color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:8px;">Filler Words Detected</div>
        <div style="margin-bottom:6px;"><span style="color:#4ade80;font-size:10px;font-weight:600;">FINE-TUNED:</span><br>{tag(ft_fillers,"#f59e0b")}</div>
        <div style="margin-top:8px;"><span style="color:#94a3b8;font-size:10px;font-weight:600;">BASE:</span><br>{tag(base_fillers,"#f59e0b")}</div>
      </div>
      <div style="background:#0f172a;border-radius:10px;padding:14px 16px;border:1px solid #1e293b;">
        <div style="color:#64748b;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-bottom:8px;">Mispronunciations Found</div>
        <div style="margin-bottom:6px;"><span style="color:#4ade80;font-size:10px;font-weight:600;">FINE-TUNED:</span><br><span style="font-size:12px;line-height:1.8;">{mispr_compact(ft_mispr)}</span></div>
        <div style="margin-top:8px;"><span style="color:#94a3b8;font-size:10px;font-weight:600;">BASE:</span><br><span style="font-size:12px;line-height:1.8;">{mispr_compact(base_mispr)}</span></div>
      </div>
    </div>
  </div>
</div>'''

cards = ''.join(compare_card(s, ft, base) for s, ft, base in zip(SESSIONS, _ft_results, _base_results))

display(HTML(f'''
<div style="font-family:'Segoe UI',sans-serif;max-width:980px;margin:0 auto;">
  <div style="background:linear-gradient(135deg,#1e1b4b,#0f172a);border-radius:14px;padding:22px 28px;margin-bottom:22px;">
    <div style="display:flex;align-items:center;gap:12px;">
      <span style="font-size:26px;">⚖️</span>
      <div>
        <div style="color:#e2e8f0;font-size:19px;font-weight:700;">Head-to-Head: Fine-tuned vs Base Model Evaluation</div>
        <div style="color:#6366f1;font-size:12.5px;margin-top:3px;">Same transcripts · Same prompt · Score comparison · Filler & mispronunciation detection accuracy</div>
      </div>
    </div>
    <div style="display:flex;gap:20px;margin-top:14px;">
      <div style="display:flex;align-items:center;gap:6px;"><div style="width:10px;height:10px;border-radius:2px;background:#166534;"></div><span style="color:#94a3b8;font-size:12px;">🧠 Fine-tuned CommuniQ — domain-aware evaluation</span></div>
      <div style="display:flex;align-items:center;gap:6px;"><div style="width:10px;height:10px;border-radius:2px;background:#334155;"></div><span style="color:#94a3b8;font-size:12px;">⚙️ Base Llama 3.1 8B — generic evaluation</span></div>
    </div>
  </div>
  {cards}
  <div style="background:#0f172a;border-radius:10px;padding:16px 20px;border:1px solid #1e293b;margin-top:4px;">
    <div style="color:#e2e8f0;font-size:13px;font-weight:600;margin-bottom:10px;">🔍 Key differences to note</div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;">
      <div>
        <div style="color:#4ade80;font-size:11px;font-weight:700;margin-bottom:6px;">Fine-tuned model:</div>
        <ul style="color:#86efac;font-size:12px;line-height:1.9;margin:0;padding-left:16px;">
          <li>Catches domain-specific mispronunciations</li>
          <li>Detects subtle filler words in Indian English</li>
          <li>Differentiates strong vs weak candidates accurately</li>
          <li>Gives targeted, actionable feedback</li>
        </ul>
      </div>
      <div>
        <div style="color:#f87171;font-size:11px;font-weight:700;margin-bottom:6px;">Base model tends to:</div>
        <ul style="color:#fca5a5;font-size:12px;line-height:1.9;margin:0;padding-left:16px;">
          <li>Miss domain-specific pronunciation errors</li>
          <li>Give inflated or deflated scores inconsistently</li>
          <li>Provide generic feedback not tailored to scenario</li>
          <li>Ignore Indian English communication patterns</li>
        </ul>
      </div>
    </div>
  </div>
</div>
'''))

In [ ]:
# @title Step 8 (OPTIONAL, redundant with 8b) -- Quick single-prompt inference test
# Numbered before 8b/8c/8d-8f existed; kept as a minimal one-off sanity check.
# Safe to skip -- Step 8b already covers this more thoroughly.
FastLanguageModel.for_inference(model)

test_messages = [
    {
        'role': 'system',
        'content': 'You are a senior HR interviewer at a tech company. Ask one behavioural question at a time. Keep replies to 2 sentences.'
    },
    {
        'role': 'assistant',
        'content': 'Walk me through a time you led a project under a tight deadline.'
    },
    {
        'role': 'user',
        'content': 'Last semester our team had to ship a fraud detection prototype in ten days. I broke it into daily milestones and we delivered on time.'
    },
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

print('Model response:')
_ = model.generate(
    input_ids=inputs,
    streamer=streamer,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9,
    use_cache=True,
)

## Step 9 — Export to GGUF for Ollama (Optional)

Run the cell below **only if** you want to use the model locally with Ollama.  
This merges the LoRA weights into the base model and quantises to Q4_K_M GGUF.

**Requirements:** ~20 GB free disk + RAM. Runs fine on Colab Pro (A100).  
On free T4 it may OOM — skip and load the adapter directly from Drive instead.

In [ ]:
# @title Step 9a — Merge + save full model (float16) — OPTIONAL
MERGED_DIR = '/content/drive/MyDrive/communiq/merged_model'
os.makedirs(MERGED_DIR, exist_ok=True)

model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f'Merged model saved to {MERGED_DIR}')

In [ ]:
# @title Step 9b — Export to GGUF Q4_K_M — OPTIONAL
import glob, shutil

GGUF_DIR = '/content/drive/MyDrive/communiq/gguf'
os.makedirs(GGUF_DIR, exist_ok=True)

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method='q4_k_m',  # good balance of size and quality
)

# Unsloth actually writes into '{GGUF_DIR}_gguf' (not GGUF_DIR itself) and names
# the file after the base model, not 'communiq'. Find whatever .gguf it produced
# and copy it to a predictable path/name so training/README.md's Step 6 and
# Modelfile.communiq.example work without needing to know this quirk.
_search_dirs = [GGUF_DIR, f'{GGUF_DIR}_gguf']
_found = []
for _d in _search_dirs:
    _found += glob.glob(f'{_d}/*.gguf')
if not _found:
    raise FileNotFoundError(f'No .gguf file found in {_search_dirs} -- check the cell output above for the real path.')
_src_gguf = max(_found, key=os.path.getsize)  # the quantized model, not any stray small file

FINAL_GGUF_PATH = f'{GGUF_DIR}/communiq-8b-q4_k_m.gguf'
if os.path.abspath(_src_gguf) != os.path.abspath(FINAL_GGUF_PATH):
    shutil.copyfile(_src_gguf, FINAL_GGUF_PATH)

size = os.path.getsize(FINAL_GGUF_PATH) / 1e9
print(f'GGUF ready at: {FINAL_GGUF_PATH}  ({size:.2f} GB)')
print("(Unsloth's raw output was left in place too, at:", _src_gguf, ")")

## Step 10 — Load into Ollama (run on your local machine)

After downloading the GGUF file from Drive:

1. Create a `Modelfile` (see `training/Modelfile.communiq.example` in the repo)
2. Run:
```bash
ollama create communiq:8b -f Modelfile
ollama run communiq:8b
```
3. Update `ai-server/.env` (classic HTTP turn-based flow):
```
LLM_BACKEND=ollama
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=communiq:8b
```
4. If also using live-voice mode, update `pipecat-worker/.env`:
```
LLM_BACKEND=openai_compat
LLM_BASE_URL=http://localhost:11434/v1
LLM_MODEL=communiq:8b
```
Restart both services after editing. Revert `LLM_BACKEND` to `groq` in either
file at any time to go back to the hosted API -- no code changes either direction.

---

## Using the LoRA adapter without full merge (lighter option)

If you don't want to export GGUF, you can load the adapter directly at inference time:

```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='/path/to/lora_output',  # the adapter dir
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

This runs on a Linux server with ~10 GB VRAM (RTX 3060 or better).